In [1]:
import copy
import sys
from pathlib import Path

import torch
from torch.export import export
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import (
    X86InductorQuantizer,
    get_default_x86_inductor_quantization_config,
)

project_root = Path.cwd()
if not (project_root / "src/cnn_project").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from cnn_project.model import EuroSATModel, model_tensor_size_mib, serialized_state_dict_size_mib, measure_inference_time_ms
from cnn_project.evaluate import evaluate_model

from cnn_project.data import create_data_loaders

device = torch.device("cpu")


W1030 10:04:15.726000 172093 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
model = EuroSATModel().cpu().eval()
model_path = project_root / "Checkpoints/base_models/model1/best.pt"
model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))

# Preserve the original model for the size comparison.
example_inputs = (torch.zeros(2, 3, 64, 64),)
exported_model = export(copy.deepcopy(model), example_inputs, dynamic_shapes=({0:torch.export.Dim("batch", min=1, max=64)},)).module()
quantizer = X86InductorQuantizer()
quantizer.set_global(get_default_x86_inductor_quantization_config())
prepared_model = prepare_pt2e(exported_model, quantizer)

# Calibrate with 200 actual training images, one at a time for this graph.
torch.manual_seed(42)
train_loader, val_loader, test_loader = create_data_loaders(
    batch_size=64, num_workers=12,
    generator=torch.Generator().manual_seed(42),
)

calibrated_images = 0
with torch.inference_mode():
    for images, _ in train_loader:
        for image in images:
            prepared_model(image.unsqueeze(0).cpu())
            calibrated_images += 1
            if calibrated_images == 200:
                break
        if calibrated_images == 200:
            break

converted_model = convert_pt2e(prepared_model, fold_quantize=True)

# Copy only attributes referenced by the graph. Conversion can leave unused
# float32 parameters alongside the new INT8 weights in the state dictionary.
quantized_model = torch.fx.GraphModule(
    converted_model, copy.deepcopy(converted_model.graph)
)
with torch.inference_mode():
    reference_output = converted_model(*example_inputs)
    output = quantized_model(*example_inputs)
    torch.testing.assert_close(output, reference_output, rtol=0, atol=0)
assert output.shape == (2, 10)
assert any(t.dtype == torch.int8 for t in quantized_model.state_dict().values())
print(f"INT8 model ready; calibrated with {calibrated_images} training images.")


INT8 model ready; calibrated with 200 training images.


In [3]:
# Compare the same measurements for both FP32 and INT8 models.
original_tensor_mib = model_tensor_size_mib(model)
quantized_tensor_mib = model_tensor_size_mib(quantized_model)
original_checkpoint_mib = serialized_state_dict_size_mib(model)
quantized_checkpoint_mib = serialized_state_dict_size_mib(quantized_model)

print("Registered tensor data (excludes serialization overhead):")
print(f"  Original (FP32):  {original_tensor_mib:.4f} MiB")
print(f"  Quantized (INT8): {quantized_tensor_mib:.4f} MiB")
print(f"  Reduction:       {100 * (1 - quantized_tensor_mib / original_tensor_mib):.1f}%")
print("Serialized state dictionary (includes file overhead):")
print(f"  Original (FP32):  {original_checkpoint_mib:.4f} MiB")
print(f"  Quantized (INT8): {quantized_checkpoint_mib:.4f} MiB")
print(f"  Reduction:       {100 * (1 - quantized_checkpoint_mib / original_checkpoint_mib):.1f}%")


Registered tensor data (excludes serialization overhead):
  Original (FP32):  0.4526 MiB
  Quantized (INT8): 0.1160 MiB
  Reduction:       74.4%
Serialized state dictionary (includes file overhead):
  Original (FP32):  0.4604 MiB
  Quantized (INT8): 0.1219 MiB
  Reduction:       73.5%


In [4]:
print("Evaluating Base Model...")
train_correct, train_total = evaluate_model(model=model,dataset_loader=train_loader, device=device)
val_correct, val_total = evaluate_model(model=model,dataset_loader=val_loader, device=device)
test_correct, test_total = evaluate_model(model=model,dataset_loader=test_loader, device=device)
print("Base Model")
print(f" Train Accuracy: {100* train_correct/train_total:.4f} %")
print(f" Val Accuracy: {100* val_correct/val_total:.4f} %")
print(f" Test Accuracy: {100* test_correct/test_total:.4f} %")


print("\nEvaluating Quantized Model...")
train_correct, train_total = evaluate_model(model=quantized_model,dataset_loader=train_loader, device=device)
val_correct, val_total = evaluate_model(model=quantized_model,dataset_loader=val_loader, device=device)
test_correct, test_total = evaluate_model(model=quantized_model,dataset_loader=test_loader, device=device)
print("Quantized Model")
print(f" Train Accuracy: {100* train_correct/train_total:.4f} %")
print(f" Val Accuracy: {100* val_correct/val_total:.4f} %")
print(f" Test Accuracy: {100* test_correct/test_total:.4f} %")

Evaluating Base Model...
Base Model
 Train Accuracy: 97.4398 %
 Val Accuracy: 96.3333 %
 Test Accuracy: 95.2963 %

Evaluating Quantized Model...
Quantized Model
 Train Accuracy: 97.2731 %
 Val Accuracy: 96.1852 %
 Test Accuracy: 95.2593 %


In [7]:
images, _ = next(iter(val_loader))
images = images.cpu()

compiled_original = torch.compile(model)
compiled_quantized = torch.compile(quantized_model)

original_ms = measure_inference_time_ms(compiled_original, images)
quantized_ms = measure_inference_time_ms(compiled_quantized, images)

print(f"Original:  {original_ms:.3f} ms/batch")
print(f"Quantized: {quantized_ms:.3f} ms/batch")
print(f"Speedup:   {original_ms / quantized_ms:.2f}×")

Original:  21.428 ms/batch
Quantized: 28.302 ms/batch
Speedup:   0.76×
